Discovery Phase (Environment Initialization & Imports)

In [7]:
pip install transformers torch sentencepiece

In [8]:
import sys
import os

# 1. Environment Verification & Installation
# Run these commands if dependencies are missing in your workspace:
# !pip install transformers torch sentencepiece

import torch
import transformers
from transformers import pipeline

print(f"Python Version: {sys.version}")
print(f"Torch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print("Discovery Phase: Environment is verified and compatible.")

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch Version: 2.11.0+cpu
Transformers Version: 5.10.2
Discovery Phase: Environment is verified and compatible.


Technical Phase – Part 1: Sentiment Analysis & Nuance Audit

In [9]:
# 1. Initialize Sentiment Pipeline
# Uses default distilbert-base-uncased-finetuned-sst-2-english
sentiment_analyzer = pipeline("sentiment-analysis")

# 2. Audit Suite: 5 Test sentences including heavy nuance/sarcasm
audit_sentences = [
    "This customer support platform completely transformed our response metrics overnight!",
    "I've been waiting three hours for a response, which is just absolutely wonderful.", # Sarcasm
    "The software works fine, but the user interface feels incredibly dated.",
    "Do not buy this product; it crashed within five minutes of setup.",
    "It is an average tool—does what it says on the box, nothing more."
]

print("=== SENTIMENT ANALYSIS AUDIT RESULTS ===")
for sentence in audit_sentences:
    result = sentiment_analyzer(sentence)[0]
    print(f"\nText: \"{sentence}\"")
    print(f"Predicted Label: {result['label']} | Confidence Score: {result['score']:.4f}")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== SENTIMENT ANALYSIS AUDIT RESULTS ===

Text: "This customer support platform completely transformed our response metrics overnight!"
Predicted Label: POSITIVE | Confidence Score: 0.9991

Text: "I've been waiting three hours for a response, which is just absolutely wonderful."
Predicted Label: POSITIVE | Confidence Score: 0.9999

Text: "The software works fine, but the user interface feels incredibly dated."
Predicted Label: NEGATIVE | Confidence Score: 0.9995

Text: "Do not buy this product; it crashed within five minutes of setup."
Predicted Label: NEGATIVE | Confidence Score: 0.9997

Text: "It is an average tool—does what it says on the box, nothing more."
Predicted Label: NEGATIVE | Confidence Score: 0.9929


Technical Phase – Part 2: Text Generation ("The Creator")

In [10]:
# 1. Initialize Text Generation Pipeline (Uses default GPT-2)
text_generator = pipeline("text-generation", model="gpt2")

# 2. Domain-Specific Prompt with Strict Length Constraints
prompt = "Customer Service Ticket: The user cannot log into their dashboard after resetting their password. Immediate Action Required:"

# Set seed for reproducible notebook execution outputs
set_seed_fn = transformers.set_seed(42)

outputs = text_generator(
    prompt,
    max_length=50,
    num_return_sequences=1,
    truncation=True
)

print("=== GENERATED RESPONSE OUTPUT ===")
print(outputs[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== GENERATED RESPONSE OUTPUT ===
Customer Service Ticket: The user cannot log into their dashboard after resetting their password. Immediate Action Required: The user must complete the following steps: Create an account, login, and then log in to the site. Create a new account on the site. Login and login to the site. Once authenticated, log into the website as a user. Access the site using the web browser. In the dashboard, select the new website. Then select a Web Hosted App. When prompted, select the Web Hosted App. If the Web Hosted App is blank, you can click the Refresh button. Click Finish to resume the page.

Step 4: Ensure that your website is up to date

Create a new web account and then log in to the site. When prompted, click the Refresh button. Once logged in to the site, click the Refresh button again and click the Apply button. You can then choose the web hosting provider to use to register your Web Hosted App.

Step 5: Create the app

In the app section, type the follo

Technical Phase – Part 3: Executive Summary Pipeline

In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Manually initialize a generative tokenizer and model
# (This completely evades the pipeline task string registry error)
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. Setup the News Article and construct an explicit summary prompt
news_article = """Artificial intelligence solutions are rapidly reshaping global corporate infrastructure.
A recent industry survey indicated that over seventy percent of tech firms are planning
to implement large language models directly into their client-facing operations by the end of the fiscal year.
While automated workflows promise a massive reduction in support queue wait times, legacy engineering executives
warn that businesses must build strict evaluation boundaries. Without human-in-the-loop validation checkpoints,
generative systems remain highly vulnerable to context drift, data leaks, and unpredictable factual hallucinations."""

# Craft an explicit prompt to instruct the causal model to summarize
prompt = f"{news_article}\n\ntl;dr summary of the text above in one concise sentence:"

# 3. Tokenize input strings into tensor matrices
inputs = tokenizer.encode(prompt, return_tensors="pt")

# 4. Generate constrained output text tokens
with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=40,    # Strictly constrain output size to 20-30 words
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

# 5. Decode tokens back into human-readable text
full_generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract only the newly generated summary response following our prompt anchor
summary_text = full_generated_text.split("tl;dr summary of the text above in one concise sentence:")[-1].strip()

print("=== EXECUTIVE SUMMARY ===")
print(summary_text)
print(f"\nSummary Word Count: {len(summary_text.split())}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== EXECUTIVE SUMMARY ===
. . .
The world's largest and most powerful AI system is now in the process of being built. It is the first to be built in a single, unified, fully automated, self

Summary Word Count: 33
